# Création du projet FPGA

Ce notebook reprend l'export de quantification et génère le RTL spécialisé et le TCL. Il peut ensuite créer le projet Vivado ou Radiant et lancer le flow complet selon la configuration.

In [20]:
from pathlib import Path
import sys

# Renseignez ce chemin uniquement si Jupyter est lancé hors du dépôt.
PROJECT_ROOT_OVERRIDE = None  # Exemple : Path('/chemin/vers/RECIPE_MLP_FPGA')

candidates = [PROJECT_ROOT_OVERRIDE] if PROJECT_ROOT_OVERRIDE else []
candidates += [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if path and (path / 'src' / 'py' / 'fpga_targets.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        'Racine introuvable : renseignez PROJECT_ROOT_OVERRIDE avec le dossier du dépôt.'
    )

source_dir = PROJECT_ROOT / 'src' / 'py'
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import importlib
import fpga_project
fpga_project = importlib.reload(fpga_project)  # Recharge le code après modification dans Jupyter
build_fpga_project = fpga_project.build_fpga_project
validate_fpga_project_config = fpga_project.validate_fpga_project_config
from fpga_targets import TARGET_PROFILES

## Configuration unique

- `create_project=False` : génère uniquement le RTL et le TCL ;
- `create_project=True` : exécute Vivado/Radiant et crée le projet ;
- `run_flow=True` avec `create_project=True` : lance aussi synthèse, placement-routage et bitstream.

In [21]:
config = {
    'name': 'leaky_relu',
    'output_root': 'data/quantized_data',

    'target': 'ultra96',             # ultra96 or lattice
    'architecture': 'sequential',    # sequential or pipeline

    'create_project': False,        # False = RTL et TCL seulement
    'run_flow': False,              # True = synthèse, placement-routage, bitstream

    'project_name': None,            # None = nom automatique
    'tool_executable': None,         # None = recherche automatique
    'jobs': 8,
    'clean_generated': True,

    # 'tool_executable': "/home/armel/lscc/radiant/2025.2/bin/lin64/radiantc", # optionnel
    # 'tool_executable': "/tools/xilinx/2025.2/Vivado/bin/vivado",             # optionnel
}

config = validate_fpga_project_config(config)
TARGET_PROFILES[config['target']]

TargetProfile(name='ultra96', transport='axi_stream', dsp_bits=48, toolchain='vivado', device='xczu3eg-sbva484-1-e', executables=('vivado',))

## Génération et création optionnelle

Le fichier `mem_layout.json` est retrouvé automatiquement à partir de `name`, `target` et `architecture`. Ces valeurs doivent donc être identiques à celles du notebook de quantification.

In [22]:
result = build_fpga_project(config, project_root=PROJECT_ROOT)
result.summary()

{'status': 'RTL et TCL générés',
 'target': 'ultra96',
 'architecture': 'sequential',
 'transport': 'axi_stream',
 'DSP_PW': 48,
 'layout': '/home/armel/Documents/PHD/MLP-GitLab/MLP_HLS/data/quantized_data/leaky_relu/ultra96/sequential/mem_layout.json',
 'RTL généré': '/home/armel/Documents/PHD/MLP-GitLab/MLP_HLS/src/rtl/generated/src_ultra_sequential',
 'TCL': '/home/armel/Documents/PHD/MLP-GitLab/MLP_HLS/vivado/create_soc_mlp_project_vivado.tcl',
 'projet': '/home/armel/Documents/PHD/MLP-GitLab/MLP_HLS/vivado/build/SOC_MLP_ultra96_sequential',
 'commande': 'non exécutée',
 'journal': 'non créé',
 'outil': 'disponible'}

In [23]:
if result.tool_error:
    raise FileNotFoundError(result.tool_error)
if result.return_code not in (None, 0):
    log_tail = result.log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-40:]
    raise RuntimeError(
        f"Échec de {result.profile.toolchain}. Fin du journal :\n" + '\n'.join(log_tail)
    )

print(result.summary()['status'])
print(f"TCL     : {result.tcl_path}")
project_label = 'Projet' if config['create_project'] else 'Projet prévu'
print(f"{project_label} : {result.project_dir}")
if result.log_path:
    print(f"Journal : {result.log_path}")

RTL et TCL générés
TCL     : /home/armel/Documents/PHD/MLP-GitLab/MLP_HLS/vivado/create_soc_mlp_project_vivado.tcl
Projet prévu : /home/armel/Documents/PHD/MLP-GitLab/MLP_HLS/vivado/build/SOC_MLP_ultra96_sequential
